In [ ]:
import os
import matplotlib.pyplot as plt
import torch
from torch.amp import autocast
from v1_model import ContinuousMotionModel
from torch.utils.data import DataLoader
from dataset.dataset import *
import utils.utils as utils
from datetime import datetime

In [ ]:
# Get the newest model from the directory. That means find the newest folder, and then the newest file in that folder.
model_path = utils.get_latest_model_path("v1_models_sweeped/sliding_vs_normal_models/normal") # or trained_models
print(f"Model path: {model_path}")

# model_path = r"sliding_diffusion_project\v1_models_sweeped\sliding_vs_normal_models\normal\normal_seed_2__3aq6g23e"

# Load the model
model: ContinuousMotionModel = ContinuousMotionModel.load_model(model_path, utils.get_device())
model.condition_mask_probabilty = 0.0  # Disable condition mask probability for inference
model = model.to(utils.get_device())

num_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters in the model: {num_params}")

device = utils.get_device()

In [ ]:
model

In [ ]:
val_loader = DataLoader(
        GPUDataset(
            consolidated_file = "dataset/genea2023_dataset/val/main-agent/consolidated.npz",
            seq_length = 70,
            seed_length = 0,
            batch_size = 64,
            epoch_length = 30,
            loading_encoded_data = False,
            include_vel_acc_features = False,
            device = device,
        ),
        batch_size = 1,
        num_workers = 0,
        pin_memory = False,
    )

In [ ]:
def compute_average_error_matrix(predictions, ground_truths):
    
    assert predictions.shape == ground_truths.shape, "Shape mismatch between predictions and ground truths"
    
    # Compute absolute difference per element
    abs_diff = torch.abs(predictions - ground_truths)  # Shape: (batch_size, *dims)
    
    # Average over the batch dimension
    avg_error_matrix = abs_diff.mean(dim=0)  # Shape: (*dims)
    
    return avg_error_matrix

In [ ]:
def visualize_error_matrix(error_matrix, save_dir, filename="average_error.png", cmap="hot"):
    if isinstance(error_matrix, torch.Tensor):
        error_matrix = error_matrix.cpu().numpy()

    plt.figure(figsize=(6, 6))
    plt.imshow(error_matrix, cmap=cmap, interpolation='nearest')
    plt.title("Average Per-Pixel Absolute Error")
    plt.colorbar()
    plt.tight_layout()
    plt.close()



In [ ]:
def visualize_and_save_error_matrix(error_matrix, save_dir, filename="average_error.png", cmap="hot"):
    if isinstance(error_matrix, torch.Tensor):
        error_matrix = error_matrix.cpu().numpy()

    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, filename)

    plt.figure(figsize=(6, 6))
    plt.imshow(error_matrix, cmap=cmap, interpolation='nearest')
    plt.title("Average Per-Pixel Absolute Error")
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

    print(f"Saved error visualization to {save_path}")

In [ ]:
all_outputs = []
all_ground_truths = []

with torch.no_grad():
    for val_batch in val_loader:
        gesture_sequence, gesture_seed, audio_features, main_agent_id_one_hot = [
            item.squeeze(0).to(device) for item in val_batch
        ]

        print(f"Processing batch with gesture_sequence shape: {gesture_sequence.shape}, "
              f"gesture_seed shape: {gesture_seed.shape}, "
              f"audio_features shape: {audio_features.shape}, "
              f"main_agent_id_one_hot shape: {main_agent_id_one_hot.shape}")
        
        output, encoded_gesture_sequence, encoded_output, noisy_gesture_sequence = model.generate(
            gesture_sequence            = gesture_sequence,
            audio_features              = audio_features,
            main_agent_id_one_hot       = main_agent_id_one_hot,
            gesture_seed                = gesture_seed,
            gesture_sequence_is_encoded = val_loader.dataset.loading_encoded_data
        )

        print(f"Output shape: {output.shape}")
        # print(output)
        
        all_outputs.append(output.cpu())
        all_ground_truths.append(gesture_sequence.cpu())

In [ ]:
all_outputs


In [ ]:
all_ground_truths

In [ ]:
print(type(all_outputs), len(all_outputs))

# Stack all batches
all_outputs_tensor  = torch.cat(all_outputs, dim=0)                 # Shape: (total_samples, *dims)
all_ground_truths_tensor  = torch.cat(all_ground_truths, dim=0)     # Same shape

# Compute the averaged error matrix
avg_error_matrix = compute_average_error_matrix(all_outputs_tensor, all_ground_truths_tensor)

print(avg_error_matrix.shape)

In [ ]:
all_outputs.clear()
all_ground_truths.clear()

In [ ]:
avg_error_matrix

In [ ]:
import numpy as np

# Convert tensor to NumPy for plotting
matrix_np = avg_error_matrix.numpy()

matrix_np = np.rot90(matrix_np)

# Plot as heatmap 
plt.imshow(matrix_np, cmap='hot', aspect='auto')
plt.colorbar(label='Intensity')
plt.title('Heatmap of Average Absolute Error')
plt.xlabel('Pose-frames')
plt.ylabel('Features')
plt.show()

In [ ]:
matrix_np_backup = matrix_np.copy()

x = torch.from_numpy(matrix_np.copy()).float()  # Convert to tensor

collapsed = torch.mean(x, dim=0, keepdim=True)

# Plot
plt.imshow(collapsed, cmap='hot', aspect='auto')
plt.colorbar(label='Intensity')
plt.title('Heatmap of Average Absolute Error - Collapsed rows (Mean)')
plt.xlabel('Pose-frames')
plt.ylabel('Features')
plt.show()

In [ ]:
import numpy as np

# Convert tensor to NumPy for plotting
matrix_np = avg_error_matrix.numpy()

matrix_np = np.rot90(matrix_np)

# Plot as heatmap 
plt.imshow(matrix_np, cmap='inferno', aspect='auto')
plt.colorbar(label='Intensity')
plt.title('Heatmap of Average Absolute Error')
plt.xlabel('Pose-frames')
plt.ylabel('Features')
plt.show()

In [ ]:
matrix_np_backup = matrix_np.copy()

x = torch.from_numpy(matrix_np.copy()).float()  # Convert to tensor

collapsed = torch.sum(x, dim=0, keepdim=True)

# Plot
plt.imshow(collapsed, cmap='inferno', aspect='auto')
plt.colorbar(label='Intensity')
plt.title('SLIDING - Heatmap of Average Absolute Error')
plt.xlabel('Pose-frames')
plt.ylabel('Features')
plt.show()

In [ ]:
# Get the newest model from the directory. That means find the newest folder, and then the newest file in that folder.
model_path = utils.get_latest_model_path("v1_models_sweeped/sliding_vs_normal_models/normal") # or trained_models
print(f"Model path: {model_path}")

# Load the model
model: ContinuousMotionModel = ContinuousMotionModel.load_model(model_path, utils.get_device())
model.condition_mask_probabilty = 0.0  # Disable condition mask probability for inference
model = model.to(utils.get_device())

num_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters in the model: {num_params}")

device = utils.get_device()

In [ ]:
all_outputs = []
all_ground_truths = []

with torch.no_grad():
    for val_batch in val_loader:
        gesture_sequence, gesture_seed, audio_features, main_agent_id_one_hot = [
            item.squeeze(0).to(device) for item in val_batch
        ]

        print(f"Processing batch with gesture_sequence shape: {gesture_sequence.shape}, "
              f"gesture_seed shape: {gesture_seed.shape}, "
              f"audio_features shape: {audio_features.shape}, "
              f"main_agent_id_one_hot shape: {main_agent_id_one_hot.shape}")
        
        output, encoded_gesture_sequence, encoded_output, noisy_gesture_sequence = model.generate(
            gesture_sequence            = gesture_sequence,
            audio_features              = audio_features,
            main_agent_id_one_hot       = main_agent_id_one_hot,
            gesture_seed                = gesture_seed,
            gesture_sequence_is_encoded = val_loader.dataset.loading_encoded_data
        )

        print(f"Output shape: {output.shape}")
        # print(output)
        
        all_outputs.append(output.cpu())
        all_ground_truths.append(gesture_sequence.cpu())

In [ ]:
print(type(all_outputs), len(all_outputs))

# Stack all batches
all_outputs_tensor  = torch.cat(all_outputs, dim=0)                 # Shape: (total_samples, *dims)
all_ground_truths_tensor  = torch.cat(all_ground_truths, dim=0)     # Same shape

# Compute the averaged error matrix
avg_error_matrix = compute_average_error_matrix(all_outputs_tensor, all_ground_truths_tensor)

print(avg_error_matrix.shape)

In [ ]:
all_outputs.clear()
all_ground_truths.clear()

In [ ]:
import numpy as np

# Convert tensor to NumPy for plotting
matrix_np = avg_error_matrix.numpy()

matrix_np = np.rot90(matrix_np)

# Plot as heatmap 
plt.imshow(matrix_np, cmap='inferno', aspect='auto')
plt.colorbar(label='Intensity')
plt.title('Heatmap of Average Absolute Error')
plt.xlabel('Pose-frames')
plt.ylabel('Features')
plt.show()

In [ ]:
matrix_np_backup = matrix_np.copy()

x = torch.from_numpy(matrix_np.copy()).float()  # Convert to tensor

collapsed = torch.sum(x, dim=0, keepdim=True)

# Plot
plt.imshow(collapsed, cmap='hot', aspect='auto')
plt.colorbar(label='Intensity')
plt.title('SLIDING - Heatmap of Average Absolute Error')
plt.xlabel('Pose-frames')
plt.ylabel('Features')
plt.show()

In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import utils.utils as utils
from typing import Optional, Tuple

def evaluate_and_plot_errors(model_path, val_loader, device, model_type, model_name, save_dir="avg_abs_error_sliding_vs_standard/avg_abs_error_diagram_results", valid_timestep_range: Optional[Tuple[int, int]] = None):
    """
    Loads a model, runs inference on val_loader, computes average error matrix,
    and saves/plots both the full error heatmap and the collapsed (row-averaged) heatmap.

    Args:
        model_path (str): Path to the trained model.
        val_loader (DataLoader): DataLoader for validation data.
        device (torch.device): Device for inference.
        save_dir (str): Directory to save plots.
    """

    # Load model
    model = ContinuousMotionModel.load_model(model_path, utils.get_device())
    model.condition_mask_probabilty = 0.0
    model = model.to(device)

    # Print model info
    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model path: {model_path}")
    print(f"Number of parameters: {num_params}")

    number_of_noisy_tensors_to_print = 1

    # Collect predictions and ground truths
    all_outputs, all_ground_truths = [], []
    with torch.no_grad():
        for val_batch in val_loader:
            gesture_sequence, gesture_seed, audio_features, main_agent_id_one_hot = [
                item.squeeze(0).to(device) for item in val_batch
            ]

            output, *_ = model.generate(
                gesture_sequence=gesture_sequence,
                audio_features=audio_features,
                main_agent_id_one_hot=main_agent_id_one_hot,
                gesture_seed=gesture_seed,
                gesture_sequence_is_encoded=val_loader.dataset.loading_encoded_data,
                valid_timestep_range=valid_timestep_range,
                show_noised_tensor=number_of_noisy_tensors_to_print > 0
            )
            number_of_noisy_tensors_to_print -= 1

            all_outputs.append(output.cpu())
            all_ground_truths.append(gesture_sequence.cpu())

    # Stack batches
    all_outputs_tensor = torch.cat(all_outputs, dim=0)
    all_ground_truths_tensor = torch.cat(all_ground_truths, dim=0)

    # Compute average error matrix
    abs_diff = torch.abs(all_outputs_tensor - all_ground_truths_tensor)
    avg_error_matrix = abs_diff.mean(dim=0)  # (*dims)

    # Convert for plotting
    matrix_np = np.rot90(avg_error_matrix.numpy()).copy()

    os.makedirs(save_dir, exist_ok=True)

    # --- Plot 1: Full heatmap ---
    plt.figure(figsize=(6, 6))
    plt.imshow(matrix_np, cmap='hot', aspect='auto')
    plt.colorbar(label='Intensity')
    plt.title(f'Heatmap of Average Absolute Error - timesteps: {model.diffusion.number_of_timesteps if valid_timestep_range is None else valid_timestep_range}')
    plt.xlabel('Pose-frames')
    plt.ylabel('Features')
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"{model_type}_{model_name}_average_error_heatmap.png"))
    plt.show()

    # --- Plot 2: Collapsed rows ---
    collapsed = torch.mean(torch.from_numpy(matrix_np).float(), dim=0, keepdim=True)
    plt.figure(figsize=(6, 2))
    plt.imshow(collapsed, cmap='hot', aspect='auto')
    plt.colorbar(label='Intensity')
    plt.title(f'Heatmap of Average Absolute Error - Collapsed rows (Mean) - timesteps: {model.diffusion.number_of_timesteps if valid_timestep_range is None else valid_timestep_range}')
    plt.xlabel('Pose-frames')
    plt.ylabel('Features')
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"{model_type}_{model_name}_average_error_collapsed.png"))
    plt.show()

    print(f"Saved plots to {save_dir}")
    print(collapsed)
    return collapsed


In [ ]:
model = ContinuousMotionModel.load_model(r"v1_models_sweeped\sliding_vs_normal_models\normal\normal_seed_2__3aq6g23e\normal_seed_2__3aq6g23e_epoch_1101.pth", utils.get_device())
model.diffusion.number_of_timesteps

In [ ]:
val_loader = DataLoader(
        GPUDataset(
            consolidated_file = "dataset/genea2023_dataset/val/main-agent/consolidated.npz",
            seq_length = 70,
            seed_length = 8,
            batch_size = 64,
            epoch_length = 30,
            loading_encoded_data = False,
            include_vel_acc_features = False,
            device = utils.get_device(),
        ),
        batch_size = 1,
        num_workers = 0,
        pin_memory = False,
    )

In [ ]:
# intervals
intervals = [
    (90, 100),
    (80, 90),
    (70, 80),
    (60, 70),
    (50, 60),
    (40, 50),
    (30, 40),
    (20, 30),
    (10, 20),
    (1, 10)
]

error_tensors = []

for start, end in intervals:
    error_tensors.append(evaluate_and_plot_errors(model_path=r"v1_models_sweeped\sliding_vs_normal_models\normal\normal_seed_2__3aq6g23e\normal_seed_2__3aq6g23e_epoch_1101.pth", 
                            val_loader=val_loader, 
                            device=utils.get_device(), 
                            model_name="seed_2",
                            model_type="normal",
                            valid_timestep_range=(start, end))
    )

In [ ]:
sliding_error_tensor = evaluate_and_plot_errors(model_path=r"v1_models_sweeped\sliding_vs_normal_models\sliding\_sliding_diffusion_seed_3__ricv3wye\_sliding_diffusion_seed_3__ricv3wye_epoch_4701.pth", 
                         val_loader=val_loader, 
                         device=utils.get_device(), 
                         model_name="seed_2",
                         model_type="sliding",)

In [ ]:
def plot_tensors(tensor_list, sliding_error_tensor):
    """
    Plots a list of PyTorch tensors as separate lines on a single graph
    with high-contrast colors for a white background.
    
    Args:
        tensor_list (list of torch.Tensor): List of tensors where each tensor
                                            has the same number of elements.
    """
    # Strong, high-contrast colors (no white, yellow, or light tones)
    colors = [
        "#ff005c",
        "#f73e5d",
        "#fa414a",
        "#e60b09",
        "#bf1f39",
        "#b01c37", 
        "#9c1937", 
        "#821736",
        "#701537", 
        "#5e1236", 
    ]
    
    plt.figure(figsize=(8, 5))

    for idx, t in enumerate(tensor_list[::-1]):
        values = t.flatten().numpy()
        plt.plot(
            range(len(values)), 
            values, 
            label=f"t = {(idx) * 10} to t = {(idx+1) * 10}", 
            color=colors[idx % len(colors)], 
            marker=''
        )
    
    # values = sliding_error_tensor.flatten().numpy()
    # plt.plot(
    #     range(len(values)), 
    #     values, 
    #     label=f"sliding - with all its ts", 
    #     color="#6ad991", 
    #     marker=''
    # )
    
    plt.xlabel("Index")
    plt.ylabel("Value")
    plt.title("Error location for validation clean predictions starting a ts in range\nNOT INFERENCE STARTING AT PURE NOISE")
    plt.legend()
    plt.grid(True, linestyle='-', alpha=0.6)

    # Move legend outside the plot
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)

    plt.tight_layout()
    plt.show()


# Example usage:
tensors = error_tensors

plot_tensors(tensors, sliding_error_tensor)

In [ ]:
model = ContinuousMotionModel.load_model(r"v1_models_sweeped\sliding_vs_normal_models\normal\normal_seed_2__3aq6g23e\normal_seed_2__3aq6g23e_epoch_1101.pth", utils.get_device())

per_sequence_timestep = torch.randint(0, model.diffusion.number_of_timesteps, (encoded_gesture_sequence.shape[0],), device=model.device)

noisy_gesture_sequence = model.diffusion.forward(encoded_gesture_sequence, per_sequence_timestep)

plt.figure(figsize=(6, 6))
plt.imshow(noisy_gesture_sequence, cmap='Greens', aspect='auto')
plt.colorbar(label='Intensity')
plt.title(f'Heatmap of Average Absolute Error - timesteps: {model.diffusion.number_of_timesteps if valid_timestep_range is None else valid_timestep_range}')
plt.xlabel('Pose-frames')
plt.ylabel('Features')
plt.tight_layout()

In [ ]:
error_tensors

In [ ]:
evaluate_and_plot_errors(model_path=r"v1_models_sweeped\sliding_vs_normal_models\sliding\_sliding_diffusion_seed_3__ricv3wye\_sliding_diffusion_seed_3__ricv3wye_epoch_4701.pth", 
                         val_loader=val_loader, 
                         device=utils.get_device(), 
                         model_name="seed_2",
                         model_type="sliding",)